## 老Attention过时了，升级一下

In [ ]:
# 之前的attention是这么个事

import torch

class Attention(torch.nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.d_k = torch.tensor(dims)
        
        self.q_proj = torch.nn.Linear(dims, dims) 
        self.k_proj = torch.nn.Linear(dims, dims)
        self.v_proj = torch.nn.Linear(dims, dims)

        self.k_cache = None
        self.v_cache = None
        
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    def forward(self, x, attention_mask=None):
        res = x
        Q = self.q_proj(x)

        if self.k_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        q_len = Q.shape[1]
        kv_len = K.shape[1]
        causal_mask = torch.ones((q_len, kv_len), dtype=torch.bool).tril(diagonal=kv_len - q_len).unsqueeze(0)

        attention_weights = torch.einsum('bqd, bld->bql', Q, K)
        attention_weights = attention_weights / torch.sqrt(self.d_k)

        if attention_mask is not None:
            mask = causal_mask & attention_mask
        else:
            mask = causal_mask
        
        attention_weights = attention_weights.masked_fill(mask==False, -1e9)
        
        attention_weights = torch.softmax(attention_weights, dim=-1)
        
        output = torch.einsum("bql, bld->bqd", attention_weights, V)
        output = self.norm(output + res)

        res = output
        output = self.feed_forward(output)
        output = self.norm(output + res)
        
        return output

    def empty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

## 思考一下下现在这种Attention有没有什么不足

首先肯定一下这个attention机制的好，有点像我们人类做阅读，读到新词了看看以前哪些词和这个新词有关联。

那我们也从人类直觉入手去想想现在这个还有什么提升空间。

---

### 从一个例子入手

假设我们读这么一个句子：爸爸的爸爸叫什么？爸爸的爸爸叫爷爷。爸爸的妈妈叫什么？爸爸的妈妈叫___

ok我们现在收到了一个词 **“叫”**，作为纯血中国人都知道下面应该填 **“奶奶”**，ok我们分析一下从相关性的角度看为什么是填奶奶：

首先 **这个"叫"** 毫无疑问和 **前面那个“叫”** 的关系最紧密，所以这两个“叫”的行为类似，**后面大概率也是一个称呼**，那为什么称呼不是爷爷呢？完全和前面一样不是很合理吗？

这是因为我们聪明的大脑袋同时也会看到 **“爸爸的爸爸”** 和 **“爸爸的妈妈”** 的不同，这个决定了后面为什么是“奶奶”不是“爷爷”。

---

### 从例子反推需求

我们想想我们的Attention类，宏观上讲他看的是在**某个角度上**当前token与之前token的相似程度，

但我们上面这个例子看得出来，next token往往需要从多个角度共同作用的。

> 那有人可能就会问了：我们现在这种不能支持多个角度吗？

>- 可以但很难，因为QKV这三个耦合得太紧了，feature内部变量之间上互相作用的，但我们要求的多角度是希望不同角度之间尽量别互相作用。

所以我们的就要把原来的Attention多搞几个，这样就能够关注到不一样的东西

In [ ]:
# 理论形成，实践开始, 最简单的实现方法就是多来几个原来的attention就完了
import torch
from torch import nn

class MultiAttention(nn.Module):
    def __init__(self, dims, num_head):
        super().__init__()

        self.dims = dims
        self.num_head = num_head

        # 为了保持dim一致，先升纬再降纬
        self.up_prj = nn.Linear(dims, num_head*dims, bias=False)
        self.attention_layers = [Attention(dims=dims) for _ in range(num_head)]
        self.down_prj = nn.Linear(num_head*dims, dims, bias=False)

    def forward(self, x):
        x_up = self.up_prj(x)

        # split一下以示配各个attention
        x_up_list = torch.split(x_up, self.num_head, dim=self.dims)
        output = [self.attention_layers[index](x_up_list[index]) for index in range(self.num_head)]

        # 聚合一下, 这时候output的shape是(batch_size, seq_length, dims*num_head)
        output = torch.cat(output, dim=-1)
        
        # 现在又是(batch_size, seq_length, dims)了
        output = self.down_prj(output)
        assert output.shape == x.shape
        return output


In [30]:
# 测试一下
new_attention = MultiAttention(dims=10, num_head=3)
rand_input = torch.randn(10, 3, 10)

output = new_attention(rand_input)
assert output.shape==rand_input.shape
print(output.shape, rand_input.shape)

torch.Size([10, 3, 10]) torch.Size([10, 3, 10])


## 考虑一下这样的MultiAttention会有什么问题

1、很明显，这样的Attention写几个head就得要有几倍的内存占用。

2、内部多个Attention虽然是并联关系，但是执行的时候还是串联执行，相当于时间又多了几倍。

## 那针对这些问题可以怎么解决呢？

我们不另外扩展了，我们选择改原来的Attention切分掉：从*num_head变成/num_head

这样很明显的一个好处就是能保证内存占用和原来的保持一致了

但同样带来的就是每个head的特征纬度变小了，虽然总的特征纬度不变，这个也就提醒我们不能把num_head的值设置过于大。

In [60]:
# 理论形成，实践开始

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, dims, num_heads):
        super().__init__()
        self.dims = torch.tensor(dims)
        # 这个也加一下
        self.num_heads = torch.tensor(num_heads)
        # 确保一定能整除
        assert dims%num_heads == 0

        self.q_proj = torch.nn.Linear(dims, dims) 
        self.k_proj = torch.nn.Linear(dims, dims)
        self.v_proj = torch.nn.Linear(dims, dims)

        self.k_cache = None
        self.v_cache = None
        
        self.feed_forward = torch.nn.Sequential(
            torch.nn.Linear(dims, 2*dims),
            torch.nn.ReLU(),
            torch.nn.Linear(2*dims, dims)
        )

        self.norm = torch.nn.LayerNorm(dims)

    def forward(self, x, attention_mask=None):
        res = x
        Q = self.q_proj(x)

        if self.k_cache is None:
            self.k_cache = self.k_proj(x)
            self.v_cache = self.v_proj(x)
        
        else:
            self.k_cache = torch.cat([self.k_cache, self.k_proj(x)], dim=1)
            self.v_cache = torch.cat([self.v_cache, self.v_proj(x)], dim=1)

        K = self.k_cache
        V = self.v_cache

        q_len = Q.shape[1]
        kv_len = K.shape[1]

        # 这时候给划分了
        # 虽然我们q_len和kv_len也写出来了，但是我还是建议写一个-1，这是个对于不定长的最后一维的习惯问题
        # 为什么要transpose最后两纬呢？因为我们想的是把切分出来的东西当成一个独立的，把他放在seq_length前面就很符合直觉，同时也方便后面的计算。
        Q = Q.reshape(Q.shape[0], -1, self.num_heads, self.dims//self.num_heads).transpose(-2, -3)
        K = K.reshape(K.shape[0], -1, self.num_heads, self.dims//self.num_heads).transpose(-2, -3)
        V = V.reshape(V.shape[0], -1, self.num_heads, self.dims//self.num_heads).transpose(-2, -3)

        # 看一下，这里涉及到shape了，之前是三维，这时候还要不要再扩展1维？
        # 其实不用，因为我们attention_mask写的时候是按没有heads写的，所以为了后面这俩mask示配，就可以不用改。
        causal_mask = torch.ones((q_len, kv_len), dtype=torch.bool).tril(diagonal=kv_len - q_len).unsqueeze(0)

        # 这里涉及到shape了，改一改，也就是加一个heads的事
        # 这时候也就可以看出来这个heads纬度和batch纬度的作用基本是比较一致的，那么后面也就不需要单独对其进行处理。
        # 同时也可以发现之前我们需要写一个for来处理多个head，现在一个矩阵乘法就ok了
        attention_weights = torch.einsum('bhqd, bhld->bhql', Q, K)

        # 这里涉及dims了，想一想前面为什么用dims，那么很合理的就是把dims//num_heads
        attention_weights = attention_weights / torch.sqrt(self.dims//self.num_heads)

        if attention_mask is not None:
            mask = causal_mask & attention_mask
        else:
            mask = causal_mask

        attention_weights = attention_weights.masked_fill(mask==False, -1e9)
        
        attention_weights = torch.softmax(attention_weights, dim=-1)
        
        # 这里涉及到shape了，改一改矩阵乘法，这时候出来的output的shape是(batch_size, num_heads, seq_length, head_dims)
        output = torch.einsum("bhql, bhld->bhqd", attention_weights, V)

        # 现在我们要恢复成和原来的一致了，不然没法res
        output = output.transpose(-2, -3).reshape(Q.shape[0], -1, self.dims)
        output = self.norm(output + res)

        res = output
        output = self.feed_forward(output)
        output = self.norm(output + res)
        
        return output

    def empty_kv_cache(self):
        self.k_cache = None
        self.v_cache = None

In [62]:
# 测试一下
attention = MultiHeadAttention(dims=10, num_heads=2)
rand_input = torch.randn(2, 3, 10)
output = attention(rand_input)
assert output.shape==rand_input.shape
print(output.shape, rand_input.shape)

torch.Size([2, 3, 10]) torch.Size([2, 3, 10])


## ok至此，你就补全了Attention论文里的最后一环，这个类我们叫它MHA

MHA也是最最常见的一个面试八股，当然你面试的时候肯定不用搓得像我这种又要考虑kv cache又要考虑length又要考虑mask的，

当然你要是都能搓出来（譬如本人），

那，

那算你背得熟练（狗头），

纯炫技罢了。

(这节是我在无座高铁餐车上站着写的，看得出无座让人烦躁，让人想多打字吐槽)

---

我把简易版本的写在下面，背就完了，也不难。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dims, num_heads):
        super().__init__()